<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">جهت <bdi dir="ltr">Gradient</bdi> را نگه دارید، اندازه‌اش را محدود کنید</h1>
<p style="text-align:right">درس 55 از 92 · وقتی گام‌ها خیلی بزرگ یا کوچک‌اند · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">49-rate</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/49-rate.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">محدودسازی <bdi dir="ltr">Norm</bdi> را از بریدن جداگانهٔ مؤلفه‌ها و از اندازهٔ نهایی <bdi dir="ltr">update</bdi> جدا کنید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: بردار،</span> <bdi dir="ltr">Norm</bdi> و ترتیب <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> تا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">optimizer.step</code>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۱۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">بردار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">[3,4]</code> با سقف <bdi dir="ltr">Norm</bdi> برابر ۱، به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">[1,1]</code> تبدیل می‌شود یا برداری هم‌جهت با خودش؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.0))
x,y = torch.tensor([[1,2,3]]),torch.tensor([[2,3,4]])
model(x,y)[1].backward()
norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
print('norm before clipping:', norm.item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">clip_vector(gradient, limit)</code> برای بردار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">float</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">limit</code> مثبت، یک بردار تازه با همان جهت و <bdi dir="ltr">Norm</bdi> حداکثر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">limit</code> برگرداند. اگر <bdi dir="ltr">Norm</bdi> کوچک است یا بردار صفر است، مقدارها تغییر نکنند.</p>
</div>

In [ ]:
def clip_vector(gradient, limit):
    # TODO: یک ضریب مشترک برای کل بردار
    return None

In [ ]:
def test_exercise():
    source = torch.tensor([3.0,4.0])
    result = clip_vector(source,1.0)
    if result is None:
        return False
    torch.testing.assert_close(result,torch.tensor([0.6,0.8]))
    torch.testing.assert_close(source,torch.tensor([3.0,4.0]))
    torch.testing.assert_close(clip_vector(torch.tensor([0.1,0.2]),1.0),torch.tensor([0.1,0.2]))
    torch.testing.assert_close(clip_vector(torch.zeros(3),1.0),torch.zeros(3))
    torch.testing.assert_close(clip_vector(torch.tensor([-6.0,8.0]),2.0),torch.tensor([-1.2,1.6]))
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: clip_vector')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right"><bdi dir="ltr">Gradient</bdi> ثابت است؛ فقط <bdi dir="ltr">Learning Rate</bdi> را عوض کنید. این مثال <bdi dir="ltr">SGD</bdi> است، نه فرمول کامل <bdi dir="ltr">AdamW</bdi>.</p>
</div>

In [ ]:
gradient = torch.tensor([0.6,0.8])
for rate in (0.01,0.1,1.0):
    update = -rate*gradient
    print(rate, 'update norm:', update.norm().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">اگر ابتدا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">step</code> را انجام دهیم، محدودکردن <bdi dir="ltr">Gradient</bdi> دیگر آن تغییر وزن را اصلاح نمی‌کند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">clipped_sgd(value, gradient, rate, limit)</code> را برای عددهای تک‌مقداری بنویسید: اول <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">gradient</code> را به بازهٔ مجاز ببرید و سپس وزن تازه را برگردانید.</p>
</div>

In [ ]:
value, gradient, rate, limit = 2.0, 10.0, 0.1, 1.0
wrong_value = value-rate*gradient
gradient = max(-limit,min(limit,gradient))
print('late clipping leaves weight at:',wrong_value)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def clipped_sgd(value, gradient, rate, limit):
    # TODO: ترتیب محدودسازی و update
    return None

In [ ]:
def test_repair():
    result = clipped_sgd(2.0,10.0,0.1,1.0)
    if result is None:
        return False
    assert abs(result-1.9)<1e-8
    assert abs(clipped_sgd(2.0,-10.0,0.1,1.0)-2.1)<1e-8
    assert abs(clipped_sgd(2.0,0.5,0.1,1.0)-1.95)<1e-8
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: clipped_sgd')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">train.py</code> مقدار بازگشتی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">clip_grad_norm_</code> را پیش از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">step</code> ثبت می‌کند. این مقدار <bdi dir="ltr">Norm</bdi> پیش از محدودسازی است، نه اندازهٔ تغییر وزن <bdi dir="ltr">AdamW</bdi>.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">اگر <bdi dir="ltr">Norm</bdi> محدود شد ولی <bdi dir="ltr">Loss</bdi> همچنان نامتناهی بود، چرا بزرگ‌ترکردن سقف راه‌حل قابل اتکایی نیست؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/49-rate.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/49-rate.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>